# Lab 22 — Phạm Hoàng Trọng · bản chạy và đóng gói tự động

Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**. NB0–NB4 chạy trước; sau đó thử bonus miễn phí. Mỗi phần có notebook riêng giữ output. Chưa chạy GPU thì chưa có số liệu hay điểm bonus.

Mặc định lưu vào Google Drive để giữ mô hình khi Colab mất phiên. Cho phép kết nối Drive khi được hỏi. Toàn bộ bonus mất nhiều giờ và có thể vượt hạn mức T4 miễn phí. Nếu bị ngắt, mở phiên GPU mới và Run all; các phần đã thành công được bỏ qua khi cấu hình không đổi. Không bảo đảm tiếp tục giữa một lượt huấn luyện chưa hoàn thành.

Cuối mỗi phần, `Lab22_submission.zip` được cập nhật. Đọc lại phản tư tự sinh trước khi nộp; ảnh GGUF vẫn cần chụp màn hình cell theo yêu cầu rubric.

In [ ]:
STUDENT_NAME = "Phạm Hoàng Trọng"
STUDENT_COHORT = "A20-K4 · 2A202602765"
USE_DRIVE = True
DRIVE_FOLDER = "Lab22_PhamHoangTrong_2A202602765"
COMPUTE_TIER = "T4"
# Giữ đủ dữ liệu bắt buộc; không giảm xuống cấu hình chạy thử.
BONUSES = "variants,gguf,benchmark,grpo,beta"
# Chạy riêng phần lỗi: ONLY="gguf", "variants", "benchmark", "grpo", "beta".
# ONLY="core" chỉ chạy bắt buộc; ONLY="finish" chỉ tạo báo cáo + zip.
ONLY = ""
RESUME = True
# Tuỳ chọn cần tài khoản/key: để trống thì không gọi dịch vụ.
CROSS_JUDGE_PROVIDER = ""  # gemini / openai / anthropic
CROSS_JUDGE_MODEL = ""     # model ID do bạn chọn
HF_REPO_ID = ""           # tài-khoản/lab22-dpo-experimental

In [ ]:
import os, sys, json, gzip, base64, subprocess
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    WORK = Path("/content/lab22")
WORK.mkdir(parents=True, exist_ok=True)
payload = json.loads(gzip.decompress(base64.b64decode("...")))
for relative, contents in payload.items():
    path = WORK / relative
    if relative == "submission/REFLECTION.md" and path.exists():
        continue
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents, encoding="utf-8")
os.chdir(WORK)
os.environ.update(COMPUTE_TIER=COMPUTE_TIER, STUDENT_NAME=STUDENT_NAME, STUDENT_COHORT=STUDENT_COHORT, GEN_BATCH_SIZE="2", JUDGE_PROVIDER="rm")
# OOM: thêm os.environ["MAX_LEN"]="512" rồi chạy lại core; cấu hình đổi sẽ huấn luyện lại.
print("Thư mục làm việc:", WORK)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['unsloth>=2026.10.1', 'trl>=1.13,<1.14', 'transformers>=5.2,<5.18', 'peft>=0.18,<1.0', 'accelerate>=1.10,<2.0', 'bitsandbytes>=0.48,<1.0', 'datasets>=4.7,<5.0', 'matplotlib>=3.9,<4.0', 'pandas>=2.2,<4.0', 'pyarrow>=17', 'openai>=1.55,<4.0', 'anthropic>=0.40,<2.0'], "nbclient>=0.10,<1", "nbformat>=5.10,<6", "ipykernel>=6,<8"], check=True)
subprocess.run([sys.executable, "scripts/build_colab.py"], check=True)
# Kiểm tra GPU trong tiến trình riêng để không nạp transformers trước Unsloth.
subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'Hãy chọn T4 GPU'; print(torch.cuda.get_device_name(0))"], check=True)

In [ ]:
from google.colab import userdata
if CROSS_JUDGE_PROVIDER and CROSS_JUDGE_MODEL:
    key_name = {"gemini":"GEMINI_API_KEY", "openai":"OPENAI_API_KEY", "anthropic":"ANTHROPIC_API_KEY"}[CROSS_JUDGE_PROVIDER]
    os.environ[key_name] = userdata.get(key_name)
    os.environ.update(CROSS_JUDGE_PROVIDER=CROSS_JUDGE_PROVIDER, CROSS_JUDGE_MODEL=CROSS_JUDGE_MODEL)
    BONUSES += ",cross"
if HF_REPO_ID:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["HF_REPO_ID"] = HF_REPO_ID
    BONUSES += ",hub"
command = [sys.executable, "-u", "scripts/run_colab.py", "--bonuses", BONUSES]
if RESUME: command.append("--resume")
if ONLY: command += ["--only", ONLY]
result = subprocess.run(command)
print("Pipeline exit code:", result.returncode)
print("Đọc data/eval/run_status.json để biết phần nào đã thành công hoặc còn lỗi.")

## Kết quả và tải bài nộp

Zip chứa các notebook đã chạy với output, ảnh, JSON, dữ liệu preference, mã nguồn và phản tư. Không chứa trọng số hay khóa API. Giữ mô hình trong Drive nếu muốn chạy tiếp bonus. Nếu runtime bị ngắt, lấy zip trong thư mục Drive ngay cả khi chưa tới cell này. Giải nén zip vào repo, đọc/điều chỉnh phản tư theo quan sát thật, commit rồi nộp link GitHub public trên LMS. Không gọi là hoàn thành bonus nếu `run_status.json` báo lỗi.

In [ ]:
from IPython.display import display, Markdown, Image
report = WORK / "submission/REFLECTION.md"
if report.exists(): display(Markdown(report.read_text(encoding="utf-8")))
for filename in ("03b-variants.png", "07-benchmark-comparison.png", "08-grpo-reward.png", "bonus-beta-sweep.png"):
    image_path = WORK / "submission/screenshots" / filename
    if image_path.exists(): display(Image(filename=str(image_path)))
meta = WORK / "data/eval/deploy_meta.json"
if meta.exists():
    info = json.loads(meta.read_text(encoding="utf-8"))
    print("GGUF Q4_K_M:", info["gguf_path"])
    print(info["smoke_prompt"], "\nHF:\n", info["hf_answer"], "\nGGUF:\n", info["gguf_answer"])
    print("Chụp màn hình phần này và lưu submission/screenshots/06-gguf-smoke.png vào repo.")
from google.colab import files
archive = WORK / "Lab22_submission.zip"
if archive.exists(): files.download(str(archive))
else: print("Chưa có zip: kiểm tra lỗi cài đặt hoặc GPU ở các cell trước.")